In [25]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt

In [26]:
gold = pl.read_parquet("../data/raw/Gold_1d.parquet")
nifty = pl.read_parquet("../data/raw/Nifty50_1d.parquet")
usdinr = pl.read_parquet("../data/raw/USDINR_1d.parquet")

In [27]:
nifty = nifty.rename({"('Close', '^NSEI')":"Close","('High', '^NSEI')":"High","('Low', '^NSEI')":"Low","('Open', '^NSEI')":"Open","('Volume', '^NSEI')":"Volume"})
gold = gold.rename({"('Close', 'GOLDBEES.NS')":"Close","('High', 'GOLDBEES.NS')":"High","('Low', 'GOLDBEES.NS')":"Low","('Open', 'GOLDBEES.NS')":"Open","('Volume', 'GOLDBEES.NS')":"Volume"})
usdinr = usdinr.rename({"('Close', 'USDINR=X')":"Close","('High', 'USDINR=X')":"High","('Low', 'USDINR=X')":"Low","('Open', 'USDINR=X')":"Open","('Volume', 'USDINR=X')":"Volume"})


In [28]:
nifty = nifty.with_columns(
    pl.col("Close").pct_change(n=3).alias("Ret_3d"),
    pl.col("Close").pct_change(n=5).alias("Ret_5d"),
    pl.col("Close").pct_change(n=10).alias("Ret_10d"),
    pl.col("Close").pct_change(n=20).alias("Ret_20d"),
)
nifty.head()

Close,High,Low,Open,Volume,Date,Ret_3d,Ret_5d,Ret_10d,Ret_20d
f64,f64,f64,f64,i64,datetime[ns],f64,f64,f64,f64
5993.25,6006.049805,5982.0,5982.600098,0,2013-01-02 00:00:00,null,null,null,null
6009.5,6017.0,5986.549805,6015.799805,0,2013-01-03 00:00:00,null,null,null,null
6016.149902,6020.75,5981.549805,6011.950195,0,2013-01-04 00:00:00,null,null,null,null
5988.399902,6042.149902,5977.149902,6042.149902,0,2013-01-07 00:00:00,-0.000809,null,null,null
6001.700195,6007.049805,5964.399902,5983.450195,0,2013-01-08 00:00:00,-0.001298,null,null,null


In [31]:
gold = gold.with_columns(

    pl.col("Close").pct_change().alias("Ret_1d"),
    pl.col("Close").pct_change(n=3).alias("Ret_3d"),
    pl.col("Close").pct_change(n=5).alias("Ret_5d"),
    pl.col("Close").pct_change(n=20).alias("Ret_20d"),
    pl.col("Close").rolling_mean(window_size=5).alias("MA_5d"),
    pl.col("Close").rolling_mean(window_size=20).alias("MA_20d"),

)
gold = gold.with_columns(
    pl.col("Ret_1d").rolling_std(window_size=5).alias("Vol_5d"),
    pl.col("Ret_1d").rolling_std(window_size=10).alias("Vol_10d"),
    pl.col("Ret_1d").rolling_std(window_size=20).alias("Vol_20d"),
)
gold = gold.with_columns(
    
    pl.col("Ret_1d").abs().alias("Abs_Return"),
    pl.col("Ret_1d").abs().rolling_mean(window_size=5).alias("Rolling_Abs_Return"),
    (pl.col("Vol_5d")/pl.col("Vol_20d")).alias("Vol_Ratio"),
    (pl.col("MA_5d")/pl.col("MA_20d")).alias("MA_Ratio"),
    (   pl.when((pl.col("High")-pl.col("Low")) !=0)
        .then((pl.col("Close")-pl.col("Low"))/(pl.col("High")-pl.col("Low")))
        .otherwise(0.5)
        .alias("Close_Pos_Range")),
    ((pl.col("Close")-pl.col("Open"))/pl.col("Open")).alias("Intraday_Return"),
    ((pl.col("High")-pl.col("Low"))/pl.col("Close")).rolling_mean(window_size=5).alias("Rolling_Range"),

)

gold = gold.drop(["Ret_1d","MA_5d","MA_20d"])

gold.columns

['Close',
 'High',
 'Low',
 'Open',
 'Volume',
 'Date',
 'Ret_3d',
 'Ret_5d',
 'Ret_20d',
 'Vol_5d',
 'Vol_10d',
 'Vol_20d',
 'Abs_Return',
 'Rolling_Abs_Return',
 'Vol_Ratio',
 'MA_Ratio',
 'Close_Pos_Range',
 'Intraday_Return',
 'Rolling_Range']

In [32]:
gold.head()

Close,High,Low,Open,Volume,Date,Ret_3d,Ret_5d,Ret_20d,Vol_5d,Vol_10d,Vol_20d,Abs_Return,Rolling_Abs_Return,Vol_Ratio,MA_Ratio,Close_Pos_Range,Intraday_Return,Rolling_Range
f64,f64,f64,f64,i64,datetime[ns],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
29.042959,29.042959,29.042959,29.042959,0,2013-01-01 00:00:00,null,null,null,null,null,null,null,null,null,null,0.5,0.0,null
29.106859,29.106859,29.106859,29.106859,0,2013-01-02 00:00:00,null,null,null,null,null,null,0.0022,null,null,null,0.5,0.0,null
29.16897,29.16897,29.16897,29.16897,0,2013-01-03 00:00:00,null,null,null,null,null,null,0.002134,null,null,null,0.5,0.0,null
28.510229,28.510229,28.510229,28.510229,0,2013-01-04 00:00:00,-0.018343,null,null,null,null,null,0.022584,null,null,null,0.5,0.0,null
28.931511,28.931511,28.931511,28.931511,0,2013-01-07 00:00:00,-0.006024,null,null,null,null,null,0.014777,null,null,null,0.5,0.0,0.0
